In [1]:
import os
import json
import csv
import re
ds = json.load(open('/home/XXXX/XXXX/SAT-LM/data/proofd5_test.json', 'r'))

In [15]:
print(ds[5])

{'example_id': 22, 'premises': 'Miroslav Venhoda was a Czech choir instructor who specialized in the performance of Renaissance and Baroque music.\nAny choral conductor is a musician.\nSome musicians love music.\nMiroslav Venhoda published a book in 1946 called Method of Studying Gregorian Chant.', 'conclusion': 'No choral conductor specialized in the performance of Renaissance.', 'premises-FOL': 'Czech(miroslav) ∧ ChoirInstructor(miroslav) ∧ SpecializeInPerformanceOf(miroslav, renaissanceMusic) ∧ SpecializeInPerformanceOf(miroslav, baroqueMusic)\n∀x (ChoralConductor(x) → Musician(x))\n∃x ∃y ((Musician(x) → Love(x, music)) \nPublishedBook(miroslav, methodOfStudyingGregorianChant, yr1946)', 'conclusion-FOL': '∀x (ChoralConductor(x) → ¬SpecializeInPerformanceOf(x, renaissanceMusic))', 'label': 'false'}


In [61]:
s = "I went out on 1 sep 2012 and it was better than 15 jan 2012"
r = re.compile('(.*)(1 sep 2012 )(.*)')
r.sub(r'\2\1\3',s)

'1 sep 2012 I went out on and it was better than 15 jan 2012'

In [16]:
import re

# Function to convert first-order logic to desired format
def convert_to_target_format(logic_str):
    # Replace universal quantifier ∀x with ForAll([x], ...)
    const = []
    
    if '∀' in logic_str:
        for z in logic_str.split('∀'):
            # print(z)
            if len(z) == 0: continue
            # print(z[0])
            if z[0] not in const:
                const.append(z[0])

    # print(const)
    # print(const)
    for c in const:
        logic_str = logic_str.replace('∀' + c, '').strip(' ')[1:-1]
    
    # print(logic_str)
    r = re.compile("¬([^\(][^ ][^\)]*\))")
    old_logic=logic_str
    logic_str = r.sub(r"Not(\1)", logic_str)
    # print('look here:', logic_str)
    # print(logic_str)
    # print(old_logic)
    while old_logic != logic_str:
        # r = re.compile("¬([^\(][^ ][.^\(]*\))")
        old_logic=logic_str
        logic_str = r.sub(r"Not(\1)", logic_str)
        # print(logic_str)
        # print(old_logic)
    
    last_op = ''
    consec_or = 1
    consec_and = 1
    tmp_or = 1
    tmp_and = 1
    ops = ['∨', '∧', '¬', '→', '⊕']
    for c in logic_str:
        # print(c)
        if c == '∨':
            if last_op == '∨':
                tmp_or += 1
            last_op = '∨'
            # print(last_op, c, last_op == c)
        elif c in ops and c != '∨':
            tmp_or = 1
            last_op = ''
            if consec_or < tmp_or: consec_or = temp_or
            # print(c)
    if consec_or < tmp_or: consec_or = temp_or

    last_op = ''
    for c in logic_str:
        if c == '∧':
            if last_op == '∧':
                tmp_and += 1
                # print(c, logic_str)

            last_op = '∧'
            
        elif c in ops and c != '∧':
            consec_and = 1
            last_op = ''
            if consec_and < tmp_and: consec_and = tmp_and
            print(c, logic_str)
    if consec_and < tmp_and: consec_and = tmp_and

    rstr = ''
    # print(consec_and)
    if consec_and > 1:
        for m in range(consec_and+1):
            rstr += '([^ ]*\([^\(]*\)) ∧ '
        rstr = rstr[:-3]
        r = re.compile(rstr)
        rsubstr = r'And('
        for m in range(1,1+1+consec_and):
            rsubstr += '\\' + str(m) + ', '
        rsubstr = rsubstr[:-2] + ')'
        # logic_str = r.sub(r
        logic_str = r.sub(rsubstr, logic_str)
        # print(rsubstr)
    if consec_or > 1:
        for m in range(consec_or+1):
            rstr += '([^ ]*\([^\(]*\)) ∨ '
        rstr = rstr[:-3]
        r = re.compile(rstr)
        rsubstr = r'Or('
        for m in range(1, 1+1+consec_or):
            rsubstr += '\\' + str(m) + ', '
        rsubstr = rsubstr[:-2] + ')'
        # logic_str = r.sub(r
        logic_str = r.sub(rsubstr, logic_str)
        # print(rsubstr)
    # print(consec_or)
    # Replace conjunction (∧) with And
    # print(logic_str)
    r = re.compile("¬\(([^ ]*\([^\(]*\)) ∨ ([^ ]*\([^\(^)]*\))\)")
    logic_str = r.sub(r"Not(Or(\1, \2))", logic_str)
    
    r = re.compile("([^ ]*\([^\(]*\)) ∨ ([^ ]*\([^\(]*\))")

    logic_str = r.sub(r"Or(\1, \2)", logic_str)    

    r = re.compile("¬\(([^ ]*\([^\(]*\)) ∧ ([^ ]*\([^\(^)]*\))\)")
    logic_str = r.sub(r"Not(And(\1, \2))", logic_str)
    
    r = re.compile("([^ ]*\([^\(]*\)) ∧ ([^ ]*\([^\(]*\))")
    logic_str = r.sub(r"And(\1, \2)", logic_str).replace('And ', 'And')

    
    r = re.compile("¬\(([^ ]*\([^\(]*\)) ⊕ ([^ ]*\([^\(^)]*\))\)")
    logic_str = r.sub(r"Not(Xor(\1, \2))", logic_str)

    r = re.compile("^(Not\([^ ]*\([^\(]*\)\)) ⊕ (Not\([^ ]*\([^\(^)]*\)\))$")
    logic_str = r.sub(r"Xor(\1, \2)", logic_str)
    
    r = re.compile("^(Not\([^ ]*\([^\(]*\)\)) ⊕ ([^ ]*\([^\(^)]*\))$")
    logic_str = r.sub(r"Xor(\1, \2)", logic_str)

    
    r = re.compile("^([^ ]*\([^\(]*\)) ⊕ (Not\([^ ]*\([^\(^)]*\)\))$")
    logic_str = r.sub(r"Xor(\1, \2)", logic_str)

    r = re.compile("^([^ ]*\([^\(]*\)) ⊕ ([^ ]*\([^\(^)]*\))$")
    logic_str = r.sub(r"Xor(\1, \2)", logic_str)

    r = re.compile("→ ([^ ]*\([^\(]*\)) ⊕ ([^ ]*\([^\(^)]*\))$")
    logic_str = r.sub(r"→ Xor(\1, \2)", logic_str)
    
    r = re.compile("\(([^ ]*\([^\(]*\)) ⊕ ([^ ]*\([^\(^)]*\))\)")
    logic_str = r.sub(r"Xor(\1, \2)", logic_str)

    # r = re.compile("¬\((.*)\)")

    # logic_str = r.sub(r"Not(\1)", logic_str)

    r = re.compile("(.*)→(.*)")
    logic_str = r.sub(r"Implies(\1, \2)", logic_str).replace(' ,', ',').replace('  ', ' ')
    
    # logic_str = re.sub(r"∀x", "ForAll([x],", logic_str)
    
    # Replace implication (→) with Implies
    
    
    # Replace negation (¬) with Not

    
    
    # Replace disjunction (∨) with Or
  
    # Make predicates lowercase for consistency
    # logic_str = re.sub(r"\b(Eel|Fish|Tree|DisplayedIn|Plant|LivingCreature|ComplexCell|Bacteria|Animal|Multicellular|ShownIn|seaSnake|seaEel)\b", lambda m: m.group(0).lower(), logic_str)
    old_logic_str = logic_str
    r = re.compile("(.*)([a-z])([A-Z])(.*)")
    logic_str = r.sub(r"\1\2_\3\4", logic_str)
 
    while old_logic_str != logic_str:
        old_logic_str = logic_str
        r = re.compile("(.*)([a-z])([A-Z])(.*)")
        logic_str = r.sub(r"\1\2_\3\4", logic_str)
        

    
    # Handle cases where functions or predicates are followed by variables
    logic_str = re.sub(r"([a-zA-Z]+)\((\w+)\)", r"\1(\2)", logic_str)
    if len(const) != 0:
        new_str = 'ForAll(['
        for c in const:
            new_str += c + ','
        logic_str = new_str[:-1] + '], ' + logic_str + ')'
    # if 'Implies' in logic_str:
        # logic_str = logic_str.replace('Implies(', 'Implies')[:-1]
    return logic_str.replace('( ', '(').replace(' (', '(')
def extract_predicates_and_objects(logic_str):
    # Regex to match predicates and objects inside parentheses
    predicate_pattern = r"([a-zA-Z]+)\(([^)]+)\)"
    
    # Find all matches
    matches = re.findall(predicate_pattern, logic_str)
    
    extracted = []
    
    # Process the matches to separate predicate and objects
    for predicate, objects in matches:
        object_list = objects.split(",")  # Split objects by comma if there are multiple
        extracted.append((predicate, object_list))
    
    return extracted


# for d in ds:
#     if d['example_id'] == 1337:
#         logic_statements = d['premises-FOL'].split('\n') + [d['conclusion-FOL']]
#         for z in d['premises-FOL'].split('\n'):
#             print(z)
d = ds[499]
logic_statements = d['premises-FOL'].split('\n') + [d['conclusion-FOL']]
space_ops = ['∧', '⊕', '∨']

for s in range(len(logic_statements)):
    logic_statements[s] = logic_statements[s].replace('  ', ' ')
    statement = logic_statements[s]
    c = 0
    while c < (len(logic_statements[s])):
        # print(logic_statements[s])
        # if namecounter == 1:
        #     breakpoint()
        if logic_statements[s][c] in space_ops:
            # breakpoint()
            if logic_statements[s][c-1] != ' ':
                logic_statements[s] = logic_statements[s][:c] + ' ' + logic_statements[s][c:]
                # breakpoint()
            if logic_statements[s][c+1] != ' ':
                logic_statements[s] = logic_statements[s][:c+2] + ' ' + logic_statements[s][c+2:]
                # breakpoint()
        c += 1
for statement in logic_statements:
    print(statement)
print('--------------')
predicates = []
entities = []
arity = {}
# print(logic_statements)
# logic_statements.append('∀x (DisplayedIn(x, collection) → ¬(Plant(x) ⊕ LivingCreature(x)))')


    # print("-" * 50)
# print(entities, predicates)
const = []


            
# print(const)

converted_statements = [convert_to_target_format(statement) for statement in logic_statements]
for statement in converted_statements:
    if 'ForAll' in statement:
        consts = statement.split('ForAll([')[1].split(']')[0].split(',')
        # print('yeah')
        for c in consts:
            if c not in const: const.append(c)
# print(const)
for statement in logic_statements:
    extracted = extract_predicates_and_objects(statement)
    # print(extracted)
    # print(f"Logic Statement: {statement}")
    for predicate, objects in extracted:
        old_logic_str = predicate
        r = re.compile("(.*)([a-z])([A-Z])(.*)")
        logic_str = r.sub(r"\1\2_\3\4", predicate)
        while old_logic_str != logic_str:
            old_logic_str = logic_str
            r = re.compile("(.*)([a-z])([A-Z])(.*)")
            logic_str = r.sub(r"\1\2_\3\4", logic_str)
            logic_str = logic_str
        logic_str = logic_str.lower().strip(' ')

        if logic_str not in predicates:
            predicates.append(logic_str)
            arity[logic_str] = len(objects)
        # print(f"Predicate: {predicate}, Objects: {objects}")
        for o in objects:
            # print(o)
            old_logic_str = o
            r = re.compile("(.*)([a-z])([A-Z])(.*)")
            logic_str = r.sub(r"\1\2_\3\4", o)
            while old_logic_str != logic_str:
                old_logic_str = logic_str
                r = re.compile("(.*)([a-z])([A-Z])(.*)")
                logic_str = r.sub(r"\1\2_\3\4", logic_str)
            logic_str = logic_str.lower().strip(' ')
            if o in const:continue
            if logic_str not in entities:
                entities.append(logic_str)
    
# Print the converted statements
print(const, entities, predicates)

for i in range(len(converted_statements)):
    converted_statements[i] = converted_statements[i].lower().replace('implies(', 'Implies(').replace('not(', 'Not(').replace('or(', 'Or(').replace('xor(', 'Xor(').replace('xOr', 'Xor').replace('forall(', 'ForAll(').replace('and(', 'And(')

for statement in converted_statements:
    print(statement)

CargoShip(britta) ∧ Ship(britta) ∧ BuiltFor(britta, norwegians)
ImpressedIntoServiceBy(britta, germany)
∀x ∀y (Ship(x) ∧ ImpressedIntoServiceBy(x, y) → SeizedBy(x, y))
SoldTo(britta, hongkong)
∀x ∀y (SoldTo(x, hongkong) → ¬SeizedBy(x, y))
--------------
→  (Ship(x) ∧ ImpressedIntoServiceBy(x, y) → SeizedBy(x, y
→  (SoldTo(x, hongkong) → ¬SeizedBy(x, y
¬  (SoldTo(x, hongkong) → ¬SeizedBy(x, y
['x', 'y'] ['britta', 'norwegians', 'germany', 'y', 'hongkong'] ['cargo_ship', 'ship', 'built_for', 'impressed_into_service_by', 'seized_by', 'sold_to']
And(cargo_ship(britta), ship(britta), built_fOr(britta, norwegians))
impressed_into_service_by(britta, germany)
ForAll([x,y], Implies(And((ship(x), impressed_into_service_by(x, y)), seized_by(x, y))
sold_to(britta, hongkong)
ForAll([x,y], Implies((sold_to(x, hongkong), ¬seized_by(x, y))


In [3]:
consec_and

NameError: name 'consec_and' is not defined

In [338]:
import ast
print(ast.unparse(ast.parse('Implies(Xor((Not(want_to_be_addicted_to_caffeine(rina)), Not(aware_that_drug(rina, caffeine)))), Not(And(Not(want_to_be_addicted_to(rina, caffeine)), drink_regularly(rina, coffee))))')))

AttributeError: module 'ast' has no attribute 'unparse'

In [340]:
import astunparse

In [341]:
print(astunparse.unparse(ast.parse('Implies(Xor((Not(want_to_be_addicted_to_caffeine(rina)), Not(aware_that_drug(rina, caffeine)))), Not(And(Not(want_to_be_addicted_to(rina, caffeine)), drink_regularly(rina, coffee))))')))


Implies(Xor((Not(want_to_be_addicted_to_caffeine(rina)), Not(aware_that_drug(rina, caffeine)))), Not(And(Not(want_to_be_addicted_to(rina, caffeine)), drink_regularly(rina, coffee))))



In [291]:
file_str = ''
file_str += 'from z3 import *\n'
file_str += 'ThingsSort, (' 
for e in entities:
    file_str += e + ', '
file_str = file_str[:-2]
file_str += ') = EnumSort(\'ThingsSort\', ['
for e in entities:
    file_str += '\'' + e + '\', '
file_str = file_str[:-2] + '])\n'
for p in predicates:
    file_str += p + ' = Function(\'' + p + '\', '
    for r in range(arity[p]):
        file_str += 'ThingsSort, '
    file_str += 'BoolSort())\n'
# print(file_str)
for c in const:
    file_str += c + ', '
if len(const) == 1:
    file_str = file_str[:-2] + ' = Const(\''
else:
    file_str = file_str[:-2] + ' = Consts(\''

for c in const:
    file_str += c + ' '
file_str = file_str[:-1] + '\', ThingsSort)\nprecond=[]\n'

for statement in converted_statements[:-1]:
    file_str += 'precond.append(' + statement + ')\n'

file_str += 's = Solver()\ns.add(precond)\n'
file_str += 's.add(Not(' + converted_statements[-1] + '))\n'
file_str += 'if s.check() == unsat:\n    print(\'True\')\nelse:\n    print(\'False\')'
print(file_str)


from z3 import *
ThingsSort, (collection, sea_snake, sea_eel) = EnumSort('ThingsSort', ['collection', 'sea_snake', 'sea_eel'])
eel = Function('eel', ThingsSort, BoolSort())
fish = Function('fish', ThingsSort, BoolSort())
tree = Function('tree', ThingsSort, BoolSort())
displayed_in = Function('displayed_in', ThingsSort, ThingsSort, BoolSort())
plant = Function('plant', ThingsSort, BoolSort())
living_creature = Function('living_creature', ThingsSort, BoolSort())
complex_cell = Function('complex_cell', ThingsSort, BoolSort())
bacteria = Function('bacteria', ThingsSort, BoolSort())
animal = Function('animal', ThingsSort, BoolSort())
multicellular = Function('multicellular', ThingsSort, BoolSort())
shown_in = Function('shown_in', ThingsSort, ThingsSort, BoolSort())
x = Const('x', ThingsSort)
precond=[]
precond.append(ForAll([x], Implies(eel(x), fish(x))))
precond.append(ForAll([x], Implies(fish(x), Not(tree(x)))))
precond.append(ForAll([x], Implies(displayed_in(x, collection), Xor(plant(x),

In [257]:
'Hello'.lower()

'hello'

In [246]:
print(logic_statements)

['∀x (Eel(x) → Fish(x))', '∀x (Fish(x) → ¬Tree(x))', '∀x (DisplayedIn(x, collection) → Plant(x) ⊕ LivingCreature(x))', '∀x (ComplexCell(x) → ¬Bacteria(x))', '∀x (DisplayedIn(x, collection) ∧ Animal(x) → Multicellular(x))', 'ShownIn(seaSnake, collection)', 'eel(seaSnake) ∨ LivingCreature(seaSnake) ∨ ¬Plant(seaEel)', '∀x (DisplayedIn(x, collection) → ¬(Plant(x) ⊕ LivingCreature(x)))']


In [230]:
import re

def extract_predicates_and_objects(logic_str):
    # Regex to match predicates and objects inside parentheses
    predicate_pattern = r"([a-zA-Z]+)\(([^)]+)\)"
    
    # Find all matches
    matches = re.findall(predicate_pattern, logic_str)
    
    extracted = []
    
    # Process the matches to separate predicate and objects
    for predicate, objects in matches:
        object_list = objects.split(",")  # Split objects by comma if there are multiple
        extracted.append((predicate, object_list))
    
    return extracted

# List of input first-order logic statements

# Extract predicates and objects for each logic statement
for statement in logic_statements:
    extracted = extract_predicates_and_objects(statement)
    print(f"Logic Statement: {statement}")
    for predicate, objects in extracted:
        print(f"Predicate: {predicate}, Objects: {objects}")
    print("-" * 50)


Logic Statement: ∀x (Eel(x) → Fish(x))
Predicate: Eel, Objects: ['x']
Predicate: Fish, Objects: ['x']
--------------------------------------------------
Logic Statement: ∀x (Fish(x) → ¬Tree(x))
Predicate: Fish, Objects: ['x']
Predicate: Tree, Objects: ['x']
--------------------------------------------------
Logic Statement: ∀x (DisplayedIn(x, collection) → Plant(x) ⊕ LivingCreature(x))
Predicate: DisplayedIn, Objects: ['x', ' collection']
Predicate: Plant, Objects: ['x']
Predicate: LivingCreature, Objects: ['x']
--------------------------------------------------
Logic Statement: ∀x (ComplexCell(x) → ¬Bacteria(x))
Predicate: ComplexCell, Objects: ['x']
Predicate: Bacteria, Objects: ['x']
--------------------------------------------------
Logic Statement: ∀x (DisplayedIn(x, collection) ∧ Animal(x) → Multicellular(x))
Predicate: DisplayedIn, Objects: ['x', ' collection']
Predicate: Animal, Objects: ['x']
Predicate: Multicellular, Objects: ['x']
--------------------------------------------

In [107]:
for line in ds[0]['premises-FOL'].split('\n'):
    const = []
    
    if '∀' in line:
        for z in line.split('∀'):
            if len(z) == 0: continue
            # print(z[0])
            if z[0] not in const:
                const.append(z[0])
    # print(const)
    for z in line.split('('):
        if 

['x']
['x']
['x']
[]
[]


In [123]:
vars =[]
funcs = []
for line in ds[0]['premises'].split('\n'):
    print(line)
            
    
    

All people who regularly drink coffee are dependent on caffeine.
People regularly drink coffee, or they don't want to be dependent to caffeine, or both.
No one who doesn't want to be addicted to caffeine is unaware that caffeine is a drug.
Rina is either a scholar who is unconcious that caffeine is a drug, or she is not a student and is she aware that caffeine is a drug.
Rina is either a student who is dependent on caffeine, or she is not a scholar and not dependent on caffeine.


In [152]:
for d in ds:
    if d['example_id'] == 1337:
        print('ah hah')
        for z in d['premises-FOL'].split('\n'):
            print(z)
    # break

ah hah
∀x (Eel(x) → Fish(x))
∀x (Fish(x) → ¬Tree(x))
∀x (DisplayedIn(x, collection) → Plant(x) ⊕ LivingCreature(x))
∀x (ComplexCell(x) → ¬Bacteria(x))
∀x (DisplayedIn(x, collection) ∧ Animal(x) → Multicellular(x))
ShownIn(seaSnake, collection)
eel(seaSnake) ∨ LivingCreature(seaSnake) ∨ ¬Plant(seaEel)
